# 🔬 Module 05: Explainable AI with Grad-CAM
### Gradient-Weighted Class Activation Mapping for Retinal Biomarker Localization

This notebook implements:
1. PyTorch forward and backward hooks to capture feature activations and gradients.
2. Calculation of channel importance weights using Global Average Pooling.
3. ReLU activation on weighted feature map combination to generate normalized heatmap.
4. Jet colormap overlay on high-resolution OCT B-scans.
5. Pathological biomarker interpretation across **CNV, DME, DRUSEN, and NORMAL**.

In [ ]:
import os
import io
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.dataset import load_kermany_oct_data, get_transforms, CLASS_NAMES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

### 5.1 GradCAM Class Implementation

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer=None):
        self.model = model
        self.model.eval()
        for m in self.model.modules():
            if hasattr(m, "inplace"):
                m.inplace = False
        self.target_layer = target_layer or self._find_default_target_layer()
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _find_default_target_layer(self):
        if hasattr(self.model, "layer4"):
            return self.model.layer4[-1]
        elif hasattr(self.model, "features"):
            for module in reversed(list(self.model.features.modules())):
                if isinstance(module, (nn.Conv2d, nn.BatchNorm2d)):
                    return module
            return self.model.features[-1]
        else:
            last_conv = None
            for module in self.model.modules():
                if isinstance(module, nn.Conv2d):
                    last_conv = module
            if last_conv is not None:
                return last_conv
            raise ValueError("Could not automatically locate a convolutional layer for Grad-CAM.")

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach().clone()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach().clone()

        self.f_hook = self.target_layer.register_forward_hook(forward_hook)
        self.b_hook = self.target_layer.register_full_backward_hook(backward_hook)

    def remove_hooks(self):
        if hasattr(self, "f_hook") and self.f_hook is not None:
            self.f_hook.remove()
        if hasattr(self, "b_hook") and self.b_hook is not None:
            self.b_hook.remove()

    def generate_heatmap(self, input_tensor, target_class=None):
        for param in self.model.parameters():
            param.requires_grad_(True)
        self.model.zero_grad()
        output = self.model(input_tensor)
        probs = F.softmax(output, dim=1).detach().cpu().numpy()[0]
        
        pred_class = int(np.argmax(probs))
        confidence = float(probs[pred_class])
        if target_class is None:
            target_class = pred_class

        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot, retain_graph=True)

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(input_tensor.shape[2], input_tensor.shape[3]), mode="bilinear", align_corners=False)
        cam_np = cam.squeeze().cpu().numpy()

        cam_min, cam_max = cam_np.min(), cam_np.max()
        if cam_max - cam_min > 1e-8:
            heatmap = (cam_np - cam_min) / (cam_max - cam_min)
        else:
            heatmap = np.zeros_like(cam_np)

        return heatmap, pred_class, confidence, probs

    @staticmethod
    def overlay_heatmap_on_image(original_image, heatmap, alpha=0.45, colormap="jet"):
        if isinstance(original_image, Image.Image):
            pil_img = original_image.convert("RGB")
            w, h = pil_img.size
        else:
            pil_img = Image.fromarray(original_image).convert("RGB")
            w, h = pil_img.size

        import matplotlib as mpl
        try:
            cmap = mpl.colormaps[colormap]
        except AttributeError:
            import matplotlib.cm as cm
            cmap = cm.get_cmap(colormap)
        colored_heatmap = cmap(heatmap)[:, :, :3]
        colored_heatmap_uint8 = (colored_heatmap * 255).astype(np.uint8)
        heatmap_img = Image.fromarray(colored_heatmap_uint8).resize((w, h), Image.Resampling.BILINEAR)

        blended = Image.blend(pil_img, heatmap_img, alpha=alpha)
        return blended, heatmap_img

print("GradCAM class ready.")

### 5.2 Loading Pre-Trained Diagnostic Model & Generating Explainability Gallery

In [ ]:
# Load top-performing model package
pkl_path = "models/best_retinal_oct_model.pkl"
with open(pkl_path, "rb") as f:
    pkg = pickle.load(f)

model = pkg["model"].to(device)
model_name = pkg["model_name"]
print(f"Loaded {model_name.upper()} (Test Accuracy: {pkg['metrics']['Accuracy']*100:.2f}%)")

splits = load_kermany_oct_data()
_, eval_tf = get_transforms()
gradcam = GradCAM(model)

# Find 1 sample per class
target_samples = {}
for i, lbl in enumerate(splits["test_labels"]):
    if lbl not in target_samples:
        target_samples[lbl] = splits["test_images"][i]
    if len(target_samples) == 4:
        break

fig, axes = plt.subplots(4, 3, figsize=(13, 14))
for row, (lbl, img_bytes) in enumerate(sorted(target_samples.items())):
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    tensor_img = eval_tf(pil_img).unsqueeze(0).to(device)

    heatmap, pred_c, conf, probs = gradcam.generate_heatmap(tensor_img)
    blended, heat_pil = GradCAM.overlay_heatmap_on_image(pil_img, heatmap, alpha=0.45)

    c_name = CLASS_NAMES[lbl]
    p_name = CLASS_NAMES[pred_c]

    # Original
    axes[row, 0].imshow(pil_img, cmap="gray")
    axes[row, 0].set_title(f"True: {c_name} (Original OCT)", fontsize=11, fontweight="bold")
    axes[row, 0].axis("off")

    # Heatmap
    axes[row, 1].imshow(heatmap, cmap="jet")
    axes[row, 1].set_title("Grad-CAM Activation Map", fontsize=11, fontweight="bold")
    axes[row, 1].axis("off")

    # Overlay
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title(f"Pred: {p_name} ({conf*100:.1f}%) Overlay", fontsize=11, fontweight="bold")
    axes[row, 2].axis("off")

gradcam.remove_hooks()
plt.suptitle(f"Grad-CAM Retinal Layer Biomarker Localization ({model_name.upper()})", fontsize=14, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()